In [1]:
import re
import nltk
import string
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pt
from sklearn.metrics import f1_score

pd.set_option("display.max_colwidth",200)



ModuleNotFoundError: No module named 'seaborn'

In [ ]:
import pandas as pd


test = pd.read_csv('../data/test_tweets_anuFYb8.csv')
train = pd.read_csv('../data/train_E6oV3lV.csv')

train.head()



In [ ]:
train.info()

In [ ]:
train.describe()



In [ ]:
train.describe(include='object')

In [ ]:
train[train['label']==0].head(10)

In [ ]:
train.shape

In [ ]:
train['label'].value_counts()

In [ ]:
comb = pd.concat([train,test],ignore_index=True)

comb.tail(20)



In [ ]:

def remove_pattern(pattern,text):
   return re.sub(pattern,'',text)


# print(remove_pattern('@[\w]*',"i/'m ML engineer @google & @Apple"))

In [ ]:
comb['new_tweet']= np.vectorize(remove_pattern)("@\w+",comb['tweet'])

comb.head()

In [ ]:
comb['new_tweet']= np.vectorize(remove_pattern)("@[\w+]*",comb['new_tweet'])
comb.head(20)

In [ ]:
comb.head()

In [ ]:
# comb['tweet'].str.replace("#\w+","",regex=True)
# comb['tweet'].str.replace("@\w+","",regex=True)
# comb.head()

In [ ]:
# comb['new_tweet'] = comb['tweet']

comb.head()

In [ ]:
comb['new_tweet'] = comb['new_tweet'].str.replace(r"[^a-zA-Z#]"," ",regex=True)

comb.head()

In [ ]:

# comb['new_tweet'] = comb['tweet']
comb['new_tweet'] = comb['new_tweet'].str.replace(r'@[\w]+'," ",regex=True)

comb.head()

In [ ]:
comb['new_tweet'] = comb['new_tweet'].apply(lambda x: ' '.join([ w for w in x.split() if len(w) > 3]))

comb.head()

In [ ]:
comb['new_tweet'] = comb['new_tweet'].str.replace("[^a-zA-Z#]"," ",regex=True)

comb.head()

In [ ]:
import nltk
nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer

# ps = PorterStemmer()

lm = WordNetLemmatizer()

lm.lemmatize("singing")

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')
# print(nltk.data.path)

from nltk.corpus import wordnet as wn

wn.synsets("running")




In [ ]:

res = wn.synsets("running")[1]

# res.name()

# res.definition()

# res.examples()

# lemm = res.lemmas()[1]

# lemm.name()

lemm = res.lemma_names()

t1 = res.hypernyms()[0]

t1.definition()

lemm = res.lemmas()[0].antonyms()

lemm


In [ ]:
tokenize_tweet = comb['new_tweet'].apply(lambda x: x.split())

tokenize_tweet.head()

In [ ]:
tokenize_tweet = comb['new_tweet'].str.split(" ")

tokenize_tweet.head()

In [ ]:
from nltk import PorterStemmer

ps = PorterStemmer()

tokenize_tweet = tokenize_tweet.apply(lambda x: ([ps.stem(w) for w in x]))

tokenize_tweet.head()

In [ ]:

for i in range(len(tokenize_tweet)):
  tokenize_tweet[i] = " ".join(tokenize_tweet[i])

tokenize_tweet.head()

In [ ]:
comb['new_tweet'] = tokenize_tweet


In [ ]:
all_words = ' '.join(w for w in comb['new_tweet'])

all_words

In [ ]:
from wordcloud import WordCloud

all_words = ' '.join(w for w in comb['new_tweet'])

print(all_words)

wc = WordCloud(width=800, height= 800, random_state=21, max_font_size=110).generate(all_words)

pt.figure(figsize=(10,7))
pt.axis('off')
pt.imshow(wc,interpolation="lanczos")

# pt.show()

In [ ]:


normal_words = ''.join([w for w in  comb['new_tweet'][comb['label'] == 0]])

normal_words
wc = WordCloud(width=800,height=500,random_state=21,max_font_size=100).generate(normal_words)
pt.imshow(wc,interpolation="lanczos")
pt.axis('off')

In [ ]:
negative_words = ''.join([w for w in comb['new_tweet'][comb['label'] == 1]])

wc = WordCloud(width=800,height=500,random_state=21,max_font_size=100).generate(negative_words)
pt.imshow(wc,interpolation="lanczos")
pt.axis('off')

In [ ]:
def extract_hastag(s):
  hashtag_bin = []
  for i in range(len(s)):
    hashtag_bin.append(re.findall(r'#[/w]+',i))
  return hashtag_bin

In [ ]:
train.head()

In [ ]:
# HT_normal = extract_hashtage(comb['new_tweet'][comb['label'] == 0])
# HT_normal = extract_hashtag(comb.loc[comb['label'] == 0,'tweet'])

# HT_normal


In [ ]:
def extract_hashtag(s):
  hashtag_bin = []
  for i in s:
    hashtag_bin.append(re.findall(r'#[\w]+',i))
  return hashtag_bin

# HT_normal = extract_hashtag(comb['new_tweet'][comb['label'] == 0])

HT_normal = extract_hashtag(comb.loc[comb['label'] == 0,'new_tweet'])



HT_negative = comb.loc[comb['label'] == 1,"new_tweet"].str.findall('#[\w]+')

# comb.head()

HT_negative


In [ ]:
ht_normal = sum(HT_normal,[])
ht_neg = sum(HT_negative,[])

ht_normal

In [ ]:
import seaborn as sns

a = nltk.FreqDist(ht_normal)

freq_ls = pd.DataFrame({'Hashtag':list(a.keys()),'Count':list(a.values())})

freq_ls = freq_ls.nlargest(columns='Count',n=20)

pt.figure(figsize=(16,5))
sns.barplot(data=freq_ls, x = 'Hashtag', y = 'Count', palette='viridis')




In [ ]:

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = ["I am a SDE at google","I was in dubai"]

vc = CountVectorizer()

fit = vc.fit(texts)
tran = vc. transform(texts)

# print(vc.vocabulary_)
print(vc.get_feature_names_out())

print(tran.toarray())

# print(X.toarray())

In [ ]:

vc = CountVectorizer(max_df = 0.8, min_df =2, max_features = 10000, stop_words="english",)

bow = vc.fit_transform(comb['new_tweet'])

print(bow.toarray())

In [ ]:
comb.shape

In [ ]:

'''
1. TF - like cuntvector
2. IDF - find important words
3. TF*IDF - better than countvector

1. TF

[am, sde, at, google, was, in, dubai]

s1: compute TF

"I am a SDE at google"
D1 = 1 1 1 1 0 0 0

"I was in dubai"
D2 = 0 0 0 0 1 1 1

s2:compute IDF

document feq




'''

In [ ]:

texts = ["I am an SDE at google","I was in dubai"]

tf = TfidfVectorizer()


tf.fit(texts)

print(tf.get_feature_names_out())
print(tf.idf_) # fit
X = tf.transform(texts)

print(X.toarray())



In [ ]:

tf = TfidfVectorizer(min_df=1, max_df=0.9, max_features=1000, stop_words='english')

tdif = tf.fit_transform(comb['new_tweet'])

# print(tdif.toarray)

tdif.shape

In [ ]:
!pip install gensim

In [ ]:
#Word2Vec

# from gensim.models import Word2Vec

from gensim.models import Word2Vec

sentences = [
    ["i", "love", "ai"],
    ["ai", "is", "powerful"],
    ["i", "love", "python"]
]
model = Word2Vec(
    sentences,
    vector_size=100,
    window=2,
    min_count = 1,
    sg=1
)


# print(model.wv['ai'])
# print(model.wv.most_similar("love"))
# print(model.wv.similarity('python','i'))
print(model.wv.index_to_key)


In [ ]:


tokenize_tweet = comb['new_tweet'].apply(lambda x:x.split( ))

fit = Word2Vec(tokenize_tweet,vector_size=200,window=5,min_count=2, sg=1,hs=0,negative=10,workers=2, seed=34)



In [ ]:
fit.train(tokenize_tweet,total_examples=len(comb['new_tweet']),epochs=20)

In [ ]:
# fit.wv.most_similar('dinner')

# fit.wv.index_to_key

len(fit.wv['dinner'])


In [ ]:
# !git init
# !git status
# !cd drive/



In [ ]:
def word_vector(token,size):
  vec = np.zeros(size).reshape((1,size))
  count = 0
  for x in token:
    try:
      vec += fit.wv[x].reshape((1,size))
      count += 1
    except KeyError:
      pass

  if count != 0:
    vec /= count

  return vec



In [ ]:
for i in tokenize_tweet:
  print(i)

In [ ]:
word_vector_arr = np.zeros((len(tokenize_tweet),200))

for i in range(len(tokenize_tweet)):
  word_vector_arr[i] = word_vector(tokenize_tweet[i],200)

word_vec_df = pd.DataFrame(word_vector_arr)

word_vec_df.shape


In [ ]:

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

doc = [ TaggedDocument(words=["This is my story"],tags=['doc1']),
       TaggedDocument(words=["ai", "is", "powerful"],tags=['doc2'])]




In [ ]:
model = Doc2Vec(
    doc,
    vector_size=50,
    window=2,
    min_count=1,
    epochs=20
)

In [ ]:

# model.dv["doc1"]

model.dv.most_similar("doc1")

In [ ]:

from tqdm import tqdm
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
tqdm.pandas(desc="progress-bar")


In [ ]:

def add_label(twt):
  output = []
  for i,s in zip(twt.index,twt):
    output.append(TaggedDocument(s,['tweet_'+str(i)]))
  return output



In [ ]:
label_dc = add_label(tokenize_tweet)

label_dc[:15]



In [ ]:
dc_model = Doc2Vec(dm=1, dm_mean=1,vector_size=200,window=5,min_count=2,negative=7,workers=3, alpha=0.1, seed=23)

In [ ]:
dc_model.build_vocab([i for i in tqdm(label_dc)])
dc_model.train(label_dc,total_examples=len(comb['new_tweet']),epochs=15)

In [ ]:
print(model.dv.index_to_key)

In [ ]:
doc_vec_arr = np.zeros((len(tokenize_tweet),200))

for i in range(len(comb)):
    doc_vec_arr[i,:] = dc_model.dv[i].reshape((1,200))




In [ ]:
doc_vec_df = pd.DataFrame(doc_vec_arr)

doc_vec_df.shape

In [ ]:
doc_vec_df

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [ ]:
comb.shape

In [ ]:

train_bow = bow[:31962,:]
test_bow = bow[31962,:]

xtrain,xtest,ytrain,ytest = train_test_split(train_bow,train['label'],test_size=0.3,random_state=42)

lreg = LogisticRegression()

lreg.fit(xtrain,ytrain)
pre = lreg.predict_proba(xtest)
pre_int = (pre[:,1] >=0.3).astype(int)
score = f1_score(ytest,pre_int)

score

In [ ]:

train_tdf = tdif[:31962,:]
test_tdf = tdif[31962,:]

xtrain_tdf = train_tdf[ytrain.index]
xtest_tdf = train_tdf[ytest.index]

lreg.fit(xtrain_tdf,ytrain)
tdif_test = lreg.predict_proba(xtest_tdf)
tdif_pre = (tdif_test[:,1] >= 0.3).astype(int)
tdif_score = f1_score(ytest,tdif_pre)

tdif_score


In [ ]:

train_w2v = word_vec_df.iloc[:31962,:]
test_w2v = word_vec_df.iloc[31962:,:]

xtrain_w2v = train_w2v.iloc[ytrain.index]
xtest_w2v = train_w2v.iloc[ytest.index]

lreg.fit(xtrain_w2v,ytrain)

w2v_test = lreg.predict_proba(xtest_w2v)
w2v_pre = (w2v_test[:,1] >= 0.3).astype(int)
w2v_score = f1_score(ytest,w2v_pre)

w2v_score




In [ ]:

xtrain_d2v = doc_vec_df.iloc[ytrain.index]
xtest_d2v = doc_vec_df.iloc[ytest.index]

lreg.fit(xtrain_d2v,ytrain)
d2v_test = lreg.predict_proba(xtest_d2v)
d2v_pre = (d2v_test[:,1] >= 0.3).astype(int)
d2v_score = f1_score(ytest,d2v_pre)
d2v_score


In [ ]:
from sklearn.svm import SVC
from sklearn.svm import LinearSVC


# svc_model = LinearSVC(kernel='linear', C = 1, probability=True)

svc_model = SVC(kernel='linear', C = 1, probability=True)
svc_model.fit(xtrain,ytrain)
svc_test = svc_model.predict_proba(xtest)
svc_prob = (svc_test[:,1] >= 0.3).astype(int)
svc_score = f1_score(ytest,svc_prob)

svc_score

In [ ]:
train_tdf = tdif[:31962,:]
test_tdf = tdif[31962,:]

xtrain_tdf = train_tdf[ytrain.index]
xtest_tdf = train_tdf[ytest.index]

svc_model.fit(xtrain_tdf,ytrain)
tdif_test = svc_model.predict_proba(xtest_tdf)
tdif_pre = (tdif_test[:,1] >= 0.3).astype(int)
tdif_score = f1_score(ytest,tdif_pre)

tdif_score



In [ ]:
train_w2v = word_vec_df.iloc[:31962,:]
test_w2v = word_vec_df.iloc[31962:,:]

xtrain_w2v = train_w2v.iloc[ytrain.index]
xtest_w2v = train_w2v.iloc[ytest.index]

svc_model.fit(xtrain_w2v,ytrain)

w2v_test = svc_model.predict_proba(xtest_w2v)
w2v_pre = (w2v_test[:,1] >= 0.3).astype(int)
w2v_score = f1_score(ytest,w2v_pre)

w2v_score

In [ ]:

xtrain_d2v = doc_vec_df.iloc[ytrain.index]
xtest_d2v = doc_vec_df.iloc[ytest.index]

svc_model.fit(xtrain_d2v,ytrain)
d2v_test = svc_model.predict_proba(xtest_d2v)
d2v_pre = (d2v_test[:,1] >= 0.3).astype(int)
d2v_score = f1_score(ytest,d2v_pre)
d2v_score


In [ ]:
# Random forest classifier

from sklearn.ensemble import RandomForestClassifier

ranFor_model = RandomForestClassifier(n_estimators = 100, random_state = 42)
ranFor_model.fit(xtrain,ytrain)


In [ ]:

ranFor_test = ranFor_model.predict(xtest)
ranFor_score = f1_score(ytest,ranFor_test)

ranFor_score


In [ ]:
train_tdf = tdif[:31962,:]
test_tdf = tdif[31962,:]

xtrain_tdf = train_tdf[ytrain.index]
xtest_tdf = train_tdf[ytest.index]

ranFor_model.fit(xtrain_tdf,ytrain)
ranFor_test = ranFor_model.predict(xtest_tdf)
ranFor_score = f1_score(ytest,ranFor_test)

randFor_score

In [ ]:
train_w2v = word_vec_df.iloc[:31962,:]
test_w2v = word_vec_df.iloc[31962:,:]

xtrain_w2v = train_w2v.iloc[ytrain.index]
xtest_w2v = train_w2v.iloc[ytest.index]

ranFor_model.fit(xtrain_w2v,ytrain)
ranFor_test = svc_model.predict(xtest_w2v)

ranFor_score = f1_score(ytest,w2v_test)

ranFor_score